# Сравнение разметки пользователей и моделей 🦋

Загрузите CSV с разметкой пользователей и ZIP с результатами моделей. Ноутбук объединит данные по `photo_id`, рассчитает метрики и сохранит Excel.

## Основные результаты

Сравнение с пользовательской разметкой на **400 фотографиях**; совпадение рамок определяется по IoU.

| Модель | Precision @ 0,5 | Recall @ 0,5 | F1 @ 0,5 | F1 @ 0,75 | FP @ 0,5 | FN @ 0,5 |
|:--|--:|--:|--:|--:|--:|--:|
| Grounding DINO | 0,7515 | 0,8986 | 0,8185 | 0,7690 | 82 | 28 |
| DINO + очистка цветов | 0,7774 | 0,8986 | 0,8336 | 0,7832 | 71 | 28 |
| Qwen | 0,9410 | 0,9819 | 0,9610 | 0,8794 | 17 | 5 |

**Краткий вывод:** Qwen показал наибольший F1 на этой выборке. Очистка цветов уменьшила число лишних рамок DINO с 82 до 71 при неизменном числе пропусков.


## 1. Загрузка файлов
Выберите `butterflies_400_final.csv` и ZIP с результатами моделей.

In [10]:
# from google.colab import files
# from IPython.display import display
# from pathlib import Path
# import io, json, math, zipfile
# import numpy as np
# import pandas as pd

# uploaded = files.upload()
# csv_files = [f for f in uploaded if f.lower().endswith('.csv')]
# zip_files = [f for f in uploaded if f.lower().endswith('.zip')]
# if len(csv_files) != 1 or len(zip_files) != 1:
#     raise ValueError('Загрузите ровно один CSV с разметкой и один ZIP с результатами.')
# USER_FILE, MODEL_ZIP = csv_files[0], zip_files[0]
# print('Разметка:', USER_FILE, '| Архив:', MODEL_ZIP)

from google.colab import files
from IPython.display import display
from pathlib import Path
import io, json, math, zipfile

import numpy as np
import pandas as pd


# Укажи свои пути к файлам
USER_FILE = Path("/content/butterflies_400_final.csv")
MODEL_ZIP = Path("/content/butterfly_dino_qwen_results.zip")


# Проверяем, что файлы существуют
if not USER_FILE.is_file():
    raise FileNotFoundError(f"Не найден файл с разметкой: {USER_FILE}")

if not MODEL_ZIP.is_file():
    raise FileNotFoundError(f"Не найден архив моделей: {MODEL_ZIP}")

if not zipfile.is_zipfile(MODEL_ZIP):
    raise ValueError(f"Файл не является корректным ZIP-архивом: {MODEL_ZIP}")


print("Разметка:", USER_FILE)
print("Результаты моделей:", MODEL_ZIP)

Разметка: /content/butterflies_400_final.csv
Результаты моделей: /content/butterfly_dino_qwen_results.zip


## 2. Объединение и расчёт метрик
Рамки сопоставляются попарно при IoU ≥ 0,5 и 0,75. Фотографии с ошибками или отсутствующими результатами исключаются из метрик соответствующей модели.

In [11]:
# При необходимости оставьте здесь только интересующие файлы, например:
# MODEL_FILES = ['dino_400.xlsx', 'qwen_fixed_400.xlsx', 'dino_clean_400.xlsx']
MODEL_FILES = None  # None: все файлы *_400.xlsx в архиве

def ids(s):
    s = s.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
    if s.isna().any() or (s == '').any():
        raise ValueError('Есть пустые photo_id')
    return s

def parse_boxes(value, scale=None):
    if pd.isna(value) or not str(value).strip():
        raise ValueError('Нет данных о рамках')
    data = json.loads(str(value))
    if not isinstance(data, list):
        raise ValueError('Ожидался список рамок')
    result = []
    for box in data:
        if isinstance(box, dict):
            coords = [box[k] for k in ('x_min', 'y_min', 'x_max', 'y_max')]
        elif isinstance(box, (list, tuple)) and len(box) == 4:
            coords = box
        else:
            raise ValueError(f'Неподдерживаемый формат рамки: {box}')
        x1, y1, x2, y2 = map(float, coords)
        if scale is not None:
            w, h = scale
            if not all(math.isfinite(v) and v > 0 for v in (w, h)):
                raise ValueError('Не указан корректный размер изображения')
            x1, x2, y1, y2 = x1 / w, x2 / w, y1 / h, y2 / h
        if not all(math.isfinite(v) for v in (x1,y1,x2,y2)) or not (0 <= x1 < x2 <= 1.00001 and 0 <= y1 < y2 <= 1.00001):
            raise ValueError(f'Некорректные координаты: {coords}')
        result.append([x1, y1, x2, y2])
    return result

def iou(a,b):
    inter=max(0,min(a[2],b[2])-max(a[0],b[0]))*max(0,min(a[3],b[3])-max(a[1],b[1]))
    union=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union else 0

def match(gold,pred,threshold):
    # Максимальное число пар: одна рамка используется не более одного раза.
    edges=[[j for j,b in enumerate(pred) if iou(a,b)>=threshold] for a in gold]
    owner={}
    def assign(i,seen):
        for j in edges[i]:
            if j in seen: continue
            seen.add(j)
            if j not in owner or assign(owner[j],seen):
                owner[j]=i
                return True
        return False
    return sum(assign(i,set()) for i in range(len(gold)))

users=pd.read_csv(USER_FILE,dtype={'photo_id':'string'},keep_default_na=False)
if not {'photo_id','boxes','box_count'} <= set(users.columns):
    raise ValueError('В пользовательском CSV нужны photo_id, boxes и box_count')
users['photo_id']=ids(users['photo_id'])
if users.photo_id.duplicated().any(): raise ValueError('Повторяющиеся photo_id в CSV')
gold={}
for _, row in users.iterrows():
    b=parse_boxes(row['boxes'])
    if len(b)!=int(row['box_count']): raise ValueError(f"Неверное число рамок: {row['photo_id']}")
    gold[row['photo_id']]=b
combined=users.rename(columns={'boxes':'user_boxes','box_count':'user_box_count'}).copy()
metrics=[]; details=[]; reports=[]
with zipfile.ZipFile(MODEL_ZIP) as archive:
    available=[p for p in archive.namelist() if p.lower().endswith('_400.xlsx') and not Path(p).name.startswith('~$')]
    if MODEL_FILES is None: paths=sorted(available)
    else:
        paths=[]
        for filename in MODEL_FILES:
            found=[p for p in available if Path(p).name==filename]
            if len(found)!=1: raise ValueError(f'Файл {filename} найден {len(found)} раз')
            paths.append(found[0])
    if not paths: raise ValueError('В архиве нет файлов *_400.xlsx')
    if len({Path(p).stem for p in paths})!=len(paths): raise ValueError('Одинаковые имена файлов в ZIP')
    for path in paths:
        name=Path(path).stem
        data=pd.read_excel(io.BytesIO(archive.read(path)),sheet_name='images',dtype={'photo_id':'string'})
        required={'photo_id','boxes_json','width','height'}
        if not required.issubset(data.columns): raise ValueError(f'{path}: нужны {sorted(required)}, найдены {list(data.columns)}')
        data['photo_id']=ids(data['photo_id'])
        if data.photo_id.duplicated().any(): raise ValueError(f'{path}: повторяются photo_id')
        columns=[x for x in ['boxes_json','num_boxes','butterfly_detected','width','height','status','error','seconds','raw_response'] if x in data.columns]
        merged=data[['photo_id']+columns].rename(columns={col:f'{name}_{col}' for col in columns})
        combined=combined.merge(merged,on='photo_id',how='left',validate='one_to_one')
        model_by_id=data.set_index('photo_id')
        counts={t:{'tp':0,'fp':0,'fn':0} for t in (0.5,0.75)}
        checked=0; excluded=0; fp_neg=0
        for photo_id, true_boxes in gold.items():
            reason=''; pred=[]
            if photo_id not in model_by_id.index: reason='Нет результата модели'
            else:
                row=model_by_id.loc[photo_id]
                error=row.get('error',''); status=str(row.get('status','')).strip().lower()
                if (not pd.isna(error) and str(error).strip()) or status=='error': reason=str(error) if not pd.isna(error) and str(error).strip() else 'Ошибка модели'
                else:
                    try:
                        pred=parse_boxes(row['boxes_json'],(float(row['width']),float(row['height'])))
                    except Exception as exc: reason=f'Невалидный результат: {exc}'
            if reason:
                excluded+=1
                details.append({'model':name,'photo_id':photo_id,'error':reason})
                continue
            checked+=1
            for t in counts:
                tp=match(true_boxes,pred,t)
                counts[t]['tp']+=tp; counts[t]['fp']+=len(pred)-tp; counts[t]['fn']+=len(true_boxes)-tp
            if not true_boxes: fp_neg+=len(pred)
            details.append({'model':name,'photo_id':photo_id,'error':'','user_count':len(true_boxes),'model_count':len(pred),'TP@0.5':match(true_boxes,pred,0.5),'FP@0.5':len(pred)-match(true_boxes,pred,0.5),'FN@0.5':len(true_boxes)-match(true_boxes,pred,0.5)})
        item={'model':name,'photos_total':len(users),'photos_scored':checked,'photos_excluded':excluded}
        for t,v in counts.items():
            tp,fp,fn=v['tp'],v['fp'],v['fn']; suffix=str(t)
            item.update({f'TP@{suffix}':tp,f'FP@{suffix}':fp,f'FN@{suffix}':fn,
                f'precision@{suffix}':tp/(tp+fp) if tp+fp else np.nan,
                f'recall@{suffix}':tp/(tp+fn) if tp+fn else np.nan,
                f'F1@{suffix}':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else np.nan})
        item['FP_on_negative_photos']=fp_neg
        metrics.append(item)
        reports.append({'model':name,'file':path,'model_photo_ids':len(data),'matching_photo_ids':len(set(data.photo_id)&set(gold)),'extra_photo_ids':len(set(data.photo_id)-set(gold)),'excluded':excluded})
metrics_df=pd.DataFrame(metrics); details_df=pd.DataFrame(details); report_df=pd.DataFrame(reports)
print('Найдены файлы:'); display(report_df)
print('Метрики относительно пользовательской разметки:'); display(metrics_df.round(4))
print(f'Объединено фотографий: {len(combined)}')

Найдены файлы:


,model,file,model_photo_ids,matching_photo_ids,extra_photo_ids,excluded
0,dino_400,dino_400.xlsx,400,400,0,0
1,dino_clean_400,dino_clean_400.xlsx,400,400,0,0
2,flowers_400,flowers_400.xlsx,400,400,0,0
3,qwen_400,qwen_400.xlsx,400,400,0,0


Метрики относительно пользовательской разметки:


,model,photos_total,photos_scored,photos_excluded,TP@0.5,FP@0.5,FN@0.5,precision@0.5,recall@0.5,F1@0.5,TP@0.75,FP@0.75,FN@0.75,precision@0.75,recall@0.75,F1@0.75,FP_on_negative_photos
0,dino_400,400,400,0,248,82,28,0.7515,0.8986,0.8185,233,97,43,0.7061,0.8442,0.7690,71
1,dino_clean_400,400,400,0,248,71,28,0.7774,0.8986,0.8336,233,86,43,0.7304,0.8442,0.7832,60
2,flowers_400,400,400,0,148,696,128,0.1754,0.5362,0.2643,135,709,141,0.1600,0.4891,0.2411,335
3,qwen_400,400,400,0,271,17,5,0.9410,0.9819,0.9610,248,40,28,0.8611,0.8986,0.8794,11


Объединено фотографий: 400


## 3. Сохранение
Скачивается Excel с разметкой, метриками и результатами по каждой фотографии.

In [7]:
OUTPUT = 'butterflies_users_and_models.xlsx'
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    combined.to_excel(writer,sheet_name='combined',index=False)
    metrics_df.to_excel(writer,sheet_name='metrics',index=False)
    details_df.to_excel(writer,sheet_name='per_photo',index=False)
    report_df.to_excel(writer,sheet_name='merge_report',index=False)
print('Сохранено:', OUTPUT)
files.download(OUTPUT)

Сохранено: butterflies_users_and_models.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>